## 从零写一个小大模型

In [ ]:
# part 1: 导入相关的 package
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(1024)

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 1024  # 中文需要更长的上下文（原来512只够~230个中文字）
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embd: int = 768    # n_embd 也叫 hidden_dim, hidden_size
    dropout: float = 0.1
    # 中文字符级 tokenizer，词表约 13000（原 GPT-2 tokenizer 对中文极其低效！）
    vocab_size: int = 13005

    # 注意：head_size 不再是 dataclass field，
    # 改用 __post_init__ 动态计算，避免 dataclass 陷阱
    def __post_init__(self):
        self.head_size = self.n_embd // self.n_head

## 模型结构

In [ ]:
class SingleHeadAttention(nn.Module):
    # 单头注意力机制
    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.n_embd, config.head_size)
        self.value = nn.Linear(config.n_embd, config.head_size)
        self.query = nn.Linear(config.n_embd, config.head_size)
        self.head_size = config.head_size

        # 尝试学习新的写法，attention_mask 通过 register_buffer 注册
        # 因为不用计算 梯度，所以节约内存和显存，速度也更快
        self.register_buffer(
            'attention_mask', 
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            ))
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()
        k = self.key(x)
        v = self.value(x)
        q = self.query(x)
        weight = q @ k.transpose(-2, -1)   # @ 就是 torch.matmul 的简化写法
        # 一定要在 softmax 前除以 sqrt(head_size)
        weight = weight.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0, 
            float('-inf')
        ) / math.sqrt(self.head_size)  # 这里的 hidden_size 其实是 head_size，因为是单头
        weight = F.softmax(weight, dim=-1)
        weight = self.dropout(weight)
        out = weight @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads], 
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output


class FeedForward(nn.Module):
    # 实际上就是 MLP
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )
    
    def forward(self, x):
        return self.net(x)


# 接下来就是一个完整的 Block

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# 完整的  GPT model
# 注意：属性名与 train.py 保持一致 (token_emb / pos_emb)，便于加载服务器训练的 checkpoint
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # linear (4 -> 8)； weight shape 是记上是 8 * 4，
        # 所以 embedding weight 和 lm_head weight 是共享的
        # 这里学习一下 tie weight。
        # 这是为了减少参数，加快训练；（现在 25的 SLM 很多都这样做了，注意⚠️）
        # self.token_emb.weight = self.lm_head.weight

        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # 这里使用的是正态分布初始化
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 是输入的 token ids
        batch, seq_len = idx.size()
        token_emb = self.token_emb(idx)

        # seq 长度是这次输入的最大长度
        pos_emb = self.pos_emb(
            # 要确保 位置编码和输入的 idx 在同一个设备上
            torch.arange(seq_len, device=idx.device)
        )
        # 有一个经典题目：为什么 embedding 和 position 可以相加？
        x = token_emb + pos_emb   # shape is (batch, seq_len, n_embd)
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)   # shape is (batch, seq_len, vocab_size)
        
        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # 如果序列太长，只取最后 block_size 个token
            idx_cond = idx if idx.size(1) <= GPTConfig.block_size else idx[:, -GPTConfig.block_size:]
            # 获取预测
            logits, _ = self(idx_cond)
            # 只关注最后一个时间步的预测
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            # 应用softmax获取概率
            probs = F.softmax(logits, dim=-1)
            # 采样下一个token
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # 附加到序列上
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx


In [ ]:

# 写一个 dataset，为了 Dataloader 准备
class MyDataset(Dataset):
    def __init__(self, path, block_size=1024, max_lines=None):
        # 加载中文字符级 tokenizer
        import pickle
        tokenizer_path = os.path.join(
            os.path.dirname(os.path.dirname(os.getcwd())),
            "data", "tokenizer_zh_char.pkl"
        )
        with open(tokenizer_path, 'rb') as f:
            tk_data = pickle.load(f)

        self.stoi = tk_data['stoi']      # char -> id
        self.itos = tk_data['itos']      # id -> char
        self.vocab_size = tk_data['vocab_size']
        self.unk_id = tk_data['unk_id']
        self.eos_id = tk_data['eos_id']
        self.block_size = block_size

        import json
        self.encoded_data = []
        raw_data = []

        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if max_lines is not None and i >= max_lines:
                    break
                try:
                    text = json.loads(line.strip())['text']
                    raw_data.append(text)
                except (json.JSONDecodeError, Exception):
                    continue

        # 将所有文本拼接成一个长序列，用 eos 分隔
        full_encoded = []
        for text in raw_data:
            # 字符级编码：每个字一个 id
            encoded_text = [self.stoi.get(ch, self.unk_id) for ch in text]
            full_encoded.extend(encoded_text + [self.eos_id])

        # 将长序列分割成 block_size+1 的样本（x 和 y 错开一位）
        for i in range(0, len(full_encoded) - block_size, block_size):
            chunk = full_encoded[i:i + block_size + 1]
            if len(chunk) < block_size + 1:
                chunk = chunk + [self.eos_id] * (block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        """将文本编码为 token IDs（字符级）"""
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        """将 token IDs 解码为文本"""
        return ''.join(self.itos.get(i, '<unk>') for i in ids)


In [ ]:
# 数据的格式
"""
{"text":"担任地点省市的区域运营中心的办理作业。承受总部相关KPI查核。..."}
"""


# 选择数据源 ================================================
DATA_SOURCE = "wiki_zh"  # 可选: "tinystories" 或 "wiki_zh"
MAX_TRAIN_LINES = 50000       # 训练行数（数据量大时控制内存）
# ============================================================

import os, json, glob

notebook_dir = os.getcwd()
project_root = os.path.dirname(os.path.dirname(notebook_dir))

if DATA_SOURCE == "tinystories":
    # ========== 方案 A: TinyStories（英文）==========
    train_path = os.path.join(project_root, "data", "tinystories", "TinyStories-train.txt")
    val_path   = os.path.join(project_root, "data", "tinystories", "TinyStories-valid.txt")
    tmp_jsonl  = os.path.join(project_root, "data", "tinystories", "tinystories_train.jsonl")

    if not os.path.exists(tmp_jsonl):
        print(f"⏳ 转换 TinyStories txt → JSONL (最多 {MAX_TRAIN_LINES} 行)...")
        written = 0
        with open(train_path, "r", encoding="utf-8") as fin, \
             open(tmp_jsonl, "w", encoding="utf-8") as fout:
            for line in fin:
                line = line.strip()
                if len(line) >= 30:
                    fout.write(json.dumps({"text": line}, ensure_ascii=False) + "\n")
                    written += 1
                    if written >= MAX_TRAIN_LINES:
                        break
        print(f"   ✅ 已生成 {written} 条训练样本 → {tmp_jsonl}")

    print(f"📂 加载 TinyStories 数据: {tmp_jsonl}")
    full_dataset = MyDataset(tmp_jsonl, block_size=GPTConfig.block_size)

elif DATA_SOURCE == "wiki_zh":
    # ========== 方案 B: 中文 Wikipedia（使用完整数据）==========
    data_path = os.path.join(project_root, "data", "wiki_zh_full.jsonl")

    if not os.path.exists(data_path):
        print("⏳ 未找到 wiki_zh_full.jsonl，尝试从原始 wiki 文件生成...")
        wiki_dir = os.path.join(project_root, "data", "wiki_zh")
        wiki_files = sorted(glob.glob(os.path.join(wiki_dir, "**", "wiki_*"), recursive=True))
        print(f"📂 找到 {len(wiki_files)} 个原始 wiki 文件")

        written = 0
        with open(data_path, "w", encoding="utf-8") as out:
            for wf in wiki_files:
                with open(wf, "r", encoding="utf-8") as f:
                    for line in f:
                        try:
                            obj = json.loads(line.strip())
                            text = obj.get("text", "").strip()
                            if len(text) >= 50:
                                out.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
                                written += 1
                        except:
                            continue
        print(f"   ✅ 已生成 {written} 条训练样本")

    print(f"📂 加载中文 Wiki 数据: {data_path}")
    print(f"   最大行数: {MAX_TRAIN_LINES}")
    full_dataset = MyDataset(data_path, block_size=GPTConfig.block_size, max_lines=MAX_TRAIN_LINES)

else:
    raise ValueError(f"未知数据源: {DATA_SOURCE}")

# 划分训练集 / 验证集
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [0.9, 0.1]
)

train_loader = DataLoader(train_dataset, batch_size=GPTConfig.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=GPTConfig.batch_size, shuffle=False)

print(f"✅ 数据加载成功！")
print(f"   Tokenizer 词表大小: {full_dataset.vocab_size}")
print(f"   训练样本数: {len(train_dataset)}")
print(f"   验证样本数: {len(val_dataset)}")

x_sample, y_sample = train_dataset[0]
decoded = full_dataset.decode(x_sample[:50].tolist())
print(f"   样本示例: {repr(decoded[:80])}...")


## 🗂️ 准备训练数据

**本笔记本支持两种数据来源：**

| 方案 | 数据 | 说明 |
|------|------|------|
| **方案 A** 英文 | `data/tinystories/TinyStories-train.txt` | 已下载，可直接用 |
| **方案 B** 中文 | `data/wiki_zh_for_train.jsonl` | 已从原始 wiki 文件提取，可直接用 |
| **方案 C** 下载新数据 | HuggingFace `wikipedia` 数据集 | 运行下方脚本自动下载 |

**数据格式要求：** JSONL 文件，每行 `{"text": "文档内容..."}`


In [ ]:
# ============================================================
# 数据加载检查 — 检查中文(wiki_zh)和英文(TinyStories)数据
# ============================================================
import os, json, sys, glob, pickle

# 获取项目根目录
notebook_dir = os.getcwd()  # src/video/
project_root = os.path.dirname(os.path.dirname(notebook_dir))  # LLMs-Zero-to-Hero-master/

print("=" * 60)
print("📂 数据文件检查")
print("=" * 60)

# ----- 1. 检查中文数据 (wiki_zh) -----
print("\n--- [中文] Wiki 百科数据 ---")
wiki_dir = os.path.join(project_root, "data", "wiki_zh")
wiki_files = sorted(glob.glob(os.path.join(wiki_dir, "**", "wiki_*"), recursive=True))
print(f"  原始 wiki 文件数: {len(wiki_files)} 个")

# 检查完整 JSONL
full_jsonl = os.path.join(project_root, "data", "wiki_zh_full.jsonl")
if os.path.exists(full_jsonl):
    size_mb = os.path.getsize(full_jsonl) / (1024 * 1024)
    print(f"  wiki_zh_full.jsonl: {size_mb:.1f} MB")

# ----- 2. 检查中文 tokenizer -----
print("\n--- [中文] Tokenizer 检查 ---")
tk_path = os.path.join(project_root, "data", "tokenizer_zh_char.pkl")
if os.path.exists(tk_path):
    with open(tk_path, 'rb') as f:
        tk = pickle.load(f)
    print(f"  ✅ tokenizer_zh_char.pkl 已就绪")
    print(f"     词表大小: {tk['vocab_size']}")
    print(f"     特殊token: unk={tk['unk_id']}, bos={tk['bos_id']}, eos={tk['eos_id']}, pad={tk['pad_id']}")

    # 编码测试
    for test_text in ["人工智能", "晋太元中", "深度学习"]:
        ids = [tk['stoi'].get(ch, tk['unk_id']) for ch in test_text]
        decoded = ''.join(tk['itos'].get(i, '<unk>') for i in ids)
        print(f"     '{test_text}' -> {len(ids)} tokens, 解码: '{decoded}'")
else:
    print("  ⚠️ tokenizer 未找到，请先运行 build_char_tokenizer.py")

# ----- 3. 检查英文数据 (TinyStories) -----
print("\n--- [英文] TinyStories 数据 ---")
tinystories_path = os.path.join(project_root, "data", "tinystories", "TinyStories-train.txt")
if os.path.exists(tinystories_path):
    fsize = os.path.getsize(tinystories_path) / (1024 * 1024)
    print(f"  文件: {tinystories_path}")
    print(f"  大小: {fsize:.1f} MB")
else:
    print("  ❌ 文件不存在")

print("\n✅ 数据检查完成！")


## 数据准备

训练这个 GPT 模型需要大量文本数据，准备 JSONL 文件，每行格式为 `{"text": "..."}`

In [ ]:
# 安装依赖（首次运行需要）
# pip install datasets tiktoken tqdm

In [ ]:
# 下载 Wikipedia 中文语料并转为 JSONL
# 这会下载到 LLMs-Zero-to-Hero-master 主文件夹下的 data/ 目录
# 首次下载约 2GB，请耐心等待

# 如果已经在终端运行了 download_wiki_to_jsonl.py，可以跳过这步
# 这里下载一个小样本（10000 条，约 100MB 数据）用于快速测试
import sys
import os

# 将项目根目录加入路径
project_root = os.path.dirname(os.getcwd())  # src/video -> src
project_root = os.path.dirname(project_root)  # src -> LLMs-Zero-to-Hero-master
sys.path.append(project_root)

# 运行下载脚本
# 方式一：从终端运行（推荐，更稳定）
print("👉 推荐在终端运行以下命令下载完整数据集：")
print(f"    python download_wiki_to_jsonl.py")
print()
print("👉 或者下载小样本测试（10000条）：")
print(f"    python download_wiki_to_jsonl.py --num-samples 10000")
print()

# 方式二：直接在这里运行（但 datasets 下载时可能会卡住）
# 取消下面这行的注释来运行：
# %run ../download_wiki_to_jsonl.py --num-samples 5000

In [ ]:
# ============================================================
# 初始化模型 & 训练设置
# ============================================================
config = GPTConfig()
model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.2f} M")
print(f"Device: {device}")
print(f"Vocab size: {config.vocab_size}")
print(f"Block size: {config.block_size}")
print(f"Head size: {config.head_size}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)


In [ ]:
# 检查 CUDA 和 PyTorch 版本
import torch
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 版本: {torch.version.cuda}")

# 如果不可用，检查 nvidia-smi
import os
os.system("nvidia-smi 2>nul && echo NVIDIA驱动存在 || echo NVIDIA驱动不存在")

In [ ]:
# ============================================================
# 训练循环（每个 epoch 保存一次 checkpoint）
# ============================================================
import time, os

os.makedirs("checkpoints", exist_ok=True)

def train(model, optimizer, scheduler, train_loader, val_loader, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, targets=y)
        
        optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
    return total_loss

def evaluate(model, val_loader, device):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss

def save_checkpoint(model, optimizer, epoch, val_loss):
    """保存每个 epoch 的 checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
    }
    ckpt_path = f'checkpoints/model_epoch_{epoch}.pt'
    torch.save(checkpoint, ckpt_path)
    size = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f'  💾 已保存: model_epoch_{epoch}.pt ({size:.1f} MB)')
    return ckpt_path


print("=" * 60)
print("🚀 开始训练")
print(f"   模型参数量: {total_params/1e6:.2f}M")
print(f"   训练设备: {device}")
print(f"   训练样本: {len(train_dataset)}, 验证样本: {len(val_dataset)}")
print(f"   批次大小: {GPTConfig.batch_size}, 序列长度: {GPTConfig.block_size}")
print(f"   每个 epoch 约 {len(train_loader)} 个 batch")
print("=" * 60)

train_losses = []
val_losses = []

# 多训练几轮，让模型充分学习中文模式
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = evaluate(model, val_loader, device)
    epoch_time = time.time() - t0
    
    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    
    # epoch 结束：保存 checkpoint
    save_checkpoint(model, optimizer, epoch, avg_val)
    
    print(f'\n🔹 Epoch {epoch+1}/{NUM_EPOCHS} 完成')
    print(f'  用时: {epoch_time:.1f}s')
    print(f'  训练 Loss: {avg_train:.4f}')
    print(f'  验证 Loss: {avg_val:.4f}')

print("=" * 60)
print("✅ 训练完成！")
print(f"训练 Loss 历史: {[f'{l:.4f}' for l in train_losses]}")
print(f"验证 Loss 历史: {[f'{l:.4f}' for l in val_losses]}")

# 显示 checkpoints 目录内容
print(f'\n📂 checkpoints 目录:')
for f in sorted(os.listdir("checkpoints")):
    size = os.path.getsize(f"checkpoints/{f}") / (1024 * 1024)
    print(f"  {f} ({size:.1f} MB)")


In [ ]:
# ============================================================
# 加载 checkpoint 继续训练 / 查看已保存的模型
# ============================================================
import os

print("📂 checkpoints 目录内容:")
if os.path.exists("checkpoints"):
    for f in sorted(os.listdir("checkpoints")):
        size = os.path.getsize(f"checkpoints/{f}") / (1024 * 1024)
        print(f"  {f} ({size:.1f} MB)")
else:
    print("  (目录不存在)")

# 如果需要加载某个 checkpoint 继续训练，取消下面注释：
# ckpt = torch.load("checkpoints/model_epoch_0.pt")
# model.load_state_dict(ckpt['model_state_dict'])
# optimizer.load_state_dict(ckpt['optimizer_state_dict'])
# print(f"✅ 已加载: epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")


## 🔧 有监督微调 (SFT — Supervised Fine-Tuning)

预训练完成后，模型学会了语言的基本模式，但还不能很好地"对话"。
有监督微调（SFT）就是用**高质量的问答对**来教模型如何回应人类指令。

### 微调 vs 预训练的核心区别

| | 预训练 (Pretrain) | 微调 (SFT) |
|------|------|------|
| **数据格式** | 原始文本 `{"text": "..."}` | 问答对 `{"prompt": "...", "response": "..."}` |
| **Loss 计算** | 对所有 token 计算 loss | **只对 response 部分计算 loss**（prompt 部分 mask 掉） |
| **学习率** | 正常 `3e-4` | 更小 `1e-5 ~ 5e-5` |
| **目的** | 学习语言模式 | 学习遵循指令


In [ ]:
# ============================================================
# SFT Dataset — 有监督微调数据集
# ============================================================

import torch
from torch.utils.data import Dataset
import pickle, json, os

class SFTDataset(Dataset):
    """SFT 微调数据集：prompt + response 拼接，只对 response 计算 loss"""
    
    def __init__(self, path, block_size=1024, max_samples=None):
        # 加载中文字符级 tokenizer
        tokenizer_path = os.path.join(
            os.path.dirname(os.path.dirname(os.getcwd())),
            "data", "tokenizer_zh_char.pkl"
        )
        with open(tokenizer_path, 'rb') as f:
            tk_data = pickle.load(f)
        
        self.stoi = tk_data['stoi']
        self.itos = tk_data['itos']
        self.unk_id = tk_data['unk_id']
        self.eos_id = tk_data['eos_id']
        self.block_size = block_size
        
        # 加载 SFT 数据（JSONL 格式：{"prompt": "...", "response": "..."}）
        self.samples = []
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if max_samples and i >= max_samples:
                    break
                try:
                    obj = json.loads(line.strip())
                    prompt = obj.get('prompt', '')
                    response = obj.get('response', '')
                    if prompt and response:
                        self.samples.append((prompt, response))
                except:
                    continue
        
        print(f"✅ SFT 数据集加载完成：{len(self.samples)} 条问答对")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        prompt, response = self.samples[idx]
        
        # 构造输入序列: prompt + response + eos
        # 💡 可以用特殊分隔符，也可以直接拼接（简单做法）
        prompt_ids = [self.stoi.get(ch, self.unk_id) for ch in prompt]
        response_ids = [self.stoi.get(ch, self.unk_id) for ch in response]
        
        # 拼接: prompt + response + <eos>
        input_ids = prompt_ids + response_ids + [self.eos_id]
        
        # labels: prompt 部分 = -100（忽略），response 部分 = 真实 id
        labels = [-100] * len(prompt_ids) + response_ids + [self.eos_id]
        
        # 截断或填充到 block_size
        if len(input_ids) > self.block_size:
            input_ids = input_ids[:self.block_size]
            labels = labels[:self.block_size]
        else:
            pad_len = self.block_size - len(input_ids)
            input_ids = input_ids + [self.eos_id] * pad_len
            labels = labels + [-100] * pad_len
        
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long)
        )
    
    def encode(self, text):
        return [self.stoi.get(ch, self.unk_id) for ch in text]
    
    def decode(self, ids):
        return ''.join(self.itos.get(i, '<unk>') for i in ids)


# 加载 SFT 数据
sft_data_path = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "data", "sft_zh_demo.jsonl"
)

if os.path.exists(sft_data_path):
    sft_dataset = SFTDataset(sft_data_path, block_size=GPTConfig.block_size)
    # 划分训练/验证
    train_sft, val_sft = torch.utils.data.random_split(sft_dataset, [0.85, 0.15])
    sft_train_loader = DataLoader(train_sft, batch_size=4, shuffle=True)
    sft_val_loader = DataLoader(val_sft, batch_size=4, shuffle=False)
    
    print(f"   训练样本: {len(train_sft)}, 验证样本: {len(val_sft)}")
    
    # 验证：检查第一条样本的 prompt mask 是否正确
    x, y = sft_dataset[0]
    prompt_masked = (y == -100).sum().item()
    response_tokens = (y != -100).sum().item()
    print(f"   样本0: input={x.size(0)} tokens, prompt_mask={prompt_masked}, response_label={response_tokens}")
    print(f"   样本0 文本: {sft_dataset.decode(x[:50].tolist())[:80]}...")
else:
    print(f"⚠️ 未找到 SFT 数据: {sft_data_path}")
    print(f"   请先创建 sft_zh_demo.jsonl，格式: {{\"prompt\": \"...\", \"response\": \"...\"}}")


In [ ]:
# ============================================================
# 🚀 SFT 微调训练
# ============================================================

import torch.nn.functional as F
import time
import copy

def sft_forward(model, input_ids, labels):
    """
    SFT 专用 forward：对 labels 中 -100 的位置忽略 loss
    """
    batch, seq_len = input_ids.size()
    token_emb = model.token_emb(input_ids)
    pos_emb = model.pos_emb(torch.arange(seq_len, device=input_ids.device))
    x = token_emb + pos_emb
    x = model.blocks(x)
    x = model.ln_final(x)
    logits = model.lm_head(x)  # (batch, seq_len, vocab_size)
    # logits[t] 是对应预测 input_ids[t+1] 的分布
    
    # shift: 预测的是下一个 token
    shift_logits = logits[:, :-1, :].contiguous()   # (B, T-1, V) 预测 t=1..T-1
    shift_labels = labels[:, 1:].contiguous()        # (B, T-1)   真实标签 t=1..T-1
    
    # 展平
    shift_logits = shift_logits.view(-1, shift_logits.size(-1))
    shift_labels = shift_labels.view(-1)
    
    # 只取 labels != -100 的位置
    mask = shift_labels != -100
    if mask.sum() == 0:
        return logits, torch.tensor(0.0, device=input_ids.device, requires_grad=True)
    
    active_logits = shift_logits[mask]
    active_labels = shift_labels[mask]
    
    loss = F.cross_entropy(active_logits, active_labels)
    return logits, loss


# ============================================================
# 加载预训练模型 & 设置微调
# ============================================================

# 1. 重新初始化模型
config = GPTConfig()
model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 2. 加载预训练 checkpoint
ckpt_path = "checkpoints/model_epoch_3.pt"
if not os.path.exists(ckpt_path):
    ckpt_path = "checkpoints/model_epoch_0.pt"
    if not os.path.exists(ckpt_path):
        # 尝试加载 sft checkpoint
        ckpt_path = "checkpoints/sft_model_epoch_2.pt"

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
    elif 'model' in ckpt:
        model.load_state_dict(ckpt['model'])
    else:
        model.load_state_dict(ckpt)
    print(f"✅ 预训练模型加载成功！{ckpt_path}")
    print(f"   Epoch: {ckpt.get('epoch', '?')}, is_sft: {ckpt.get('is_sft', False)}")

total_params = sum(p.numel() for p in model.parameters())
print(f"   模型参数量: {total_params/1e6:.2f}M")

# 3. ====== 🌟 优化后的 SFT 超参数 ======
SFT_LR = 5e-5            # 提高学习率（原来 2e-5 → 5e-5）
SFT_EPOCHS = 10          # 增加训练轮数（原来 3 → 10）
WARMUP_RATIO = 0.1       # 10% 的步骤用来 warmup
# =======================================

optimizer = torch.optim.AdamW(model.parameters(), lr=SFT_LR, weight_decay=0.01)

# 带 warmup 的余弦学习率调度
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR

total_steps = SFT_EPOCHS * len(sft_train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)  # warmup: 从 0 线性增加到 1
    else:
        # cosine decay
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

print(f"\n🔧 SFT 微调配置（优化版）:")
print(f"   学习率: {SFT_LR} (warmup={warmup_steps} steps)")
print(f"   训练轮数: {SFT_EPOCHS}")
print(f"   训练样本: {len(train_sft)} 条（数据量小，靠多轮重复训练）")
print(f"   批次大小: 4")
print(f"   每个 epoch: {len(sft_train_loader)} batches")
print(f"   总步数: {total_steps}")

# 4. 微调训练循环（增强版：加 logging 频率控制）
os.makedirs("checkpoints", exist_ok=True)

def sft_train_epoch(model, loader, optimizer, scheduler, device, epoch, global_step):
    model.train()
    total_loss = 0
    for batch_idx, (input_ids, labels) in enumerate(loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        _, loss = sft_forward(model, input_ids, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        global_step += 1
        
        # 每 2 个 batch 打印一次（因为总共才 ~3-4 个 batch）
        if len(loader) <= 5 or batch_idx % 2 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'  Batch {batch_idx+1}/{len(loader)}, Loss: {loss.item():.4f}, LR: {current_lr:.2e}')
    
    return total_loss / len(loader), global_step

@torch.no_grad()
def sft_eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    for input_ids, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        _, loss = sft_forward(model, input_ids, labels)
        total_loss += loss.item()
    return total_loss / len(loader)

print("=" * 60)
print("🚀 开始 SFT 微调（优化版）")
print("=" * 60)

sft_train_losses = []
sft_val_losses = []
global_step = 0

for epoch in range(SFT_EPOCHS):
    t0 = time.time()
    
    # 如果数据太少，每个 epoch 重新 shuffle（DataLoader 已 shuffle=True）
    train_loss, global_step = sft_train_epoch(model, sft_train_loader, optimizer, scheduler, device, epoch, global_step)
    val_loss = sft_eval_epoch(model, sft_val_loader, device)
    elapsed = time.time() - t0
    
    sft_train_losses.append(train_loss)
    sft_val_losses.append(val_loss)
    
    # 每个 epoch 都保存（方便回退到最佳 epoch）
    sft_ckpt = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_loss': val_loss,
        'train_loss': train_loss,
        'is_sft': True,
    }
    ckpt_filename = f'checkpoints/sft_model_epoch_{epoch}.pt'
    torch.save(sft_ckpt, ckpt_filename)
    
    print(f'\n🔹 SFT Epoch {epoch+1}/{SFT_EPOCHS}')
    print(f'   用时: {elapsed:.1f}s')
    print(f'   训练 Loss: {train_loss:.4f}')
    print(f'   验证 Loss: {val_loss:.4f}')
    print(f'   💾 已保存: {ckpt_filename}')

print("=" * 60)
print("✅ SFT 微调完成！")
print(f"训练 Loss 历史: {[f'{l:.4f}' for l in sft_train_losses]}")
print(f"验证 Loss 历史: {[f'{l:.4f}' for l in sft_val_losses]}")

# 打印每个 epoch 的 loss 变化趋势
print("\n📊 Loss 变化趋势:")
for i, (tl, vl) in enumerate(zip(sft_train_losses, sft_val_losses)):
    bar = '█' * int((1.5 - tl) * 20) if tl < 1.5 else '█' * 5
    print(f"  Epoch {i+1:2d}: train={tl:.4f}  val={vl:.4f}  {bar}")


In [ ]:
# ============================================================
# 🧪 SFT 微调后测试 — 对比微调前后效果
# ============================================================

import torch
import pickle

# 加载 tokenizer
notebook_dir = os.getcwd()
project_root = os.path.dirname(os.path.dirname(notebook_dir))
tokenizer_path = os.path.join(project_root, "data", "tokenizer_zh_char.pkl")
with open(tokenizer_path, 'rb') as f:
    tk_data = pickle.load(f)

stoi = tk_data['stoi']
itos = tk_data['itos']
unk_id = tk_data['unk_id']

def encode(text):
    return [stoi.get(ch, unk_id) for ch in text]

def decode(ids):
    return ''.join(itos.get(i, '<unk>') for i in ids)


def generate_response(model, prompt, max_tokens=200, temperature=0.7, top_k=50, device="cuda"):
    """
    生成回复 — 微调后的模型应该能给出更结构化的回答
    """
    input_ids = encode(prompt)
    x = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)
    
    generated = []
    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            x_cond = x if x.size(1) <= GPTConfig.block_size else x[:, -GPTConfig.block_size:]
            logits, _ = model(x_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')
            
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            x = torch.cat((x, idx_next), dim=1)
            generated.append(idx_next.item())
            
            # 如果生成了 eos 或换行符够多，就停止
            if idx_next.item() in [tk_data['eos_id']]:
                break
    
    return decode(generated)


print("\n" + "=" * 60)
print("🧪 SFT 微调后效果测试")
print("   测试问题涵盖训练数据中的领域")
print("=" * 60)

# 测试提示 — 涵盖了 SFT 训练数据的主题
test_prompts_sft = ["人工智能是什么","什么是过拟合","如何学习人工智能","Python 是什么"]

for prompt in test_prompts_sft:
    print(f"\n💬 问题: {prompt}")
    reply = generate_response(model, prompt, max_tokens=256, temperature=0.7, top_k=40, device=device)
    print(f"🤖 回答: {reply}")
    print("-" * 40)

# 测试一些未见过的问题（泛化能力）
print("\n" + "=" * 60)
print("🌟 泛化测试：未在训练数据中的问题")
print("=" * 60)

generalization_prompts = [
    "什么是强化学习",
    "Linux 和 Windows 有什么区别",
    "如何入门编程",
]

for prompt in generalization_prompts:
    print(f"\n💬 问题: {prompt}")
    reply = generate_response(model, prompt, max_tokens=256, temperature=0.7, top_k=40, device=device)
    print(f"🤖 回答: {reply}")
    print("-" * 40)

print("\n✅ SFT 测试完成！")


In [ ]:
# ============================================================
# 🤖 聊天测试 — 加载训练好的模型进行对话
# ============================================================
import torch
import pickle
import os

# ---- 1. 加载 GPT 模型配置 ----
# 如果 kernel 还是活的，model 变量应该已存在；
# 否则取消下方false，重新构建模型：

if False:
    from dataclasses import dataclass
    @dataclass
    class GPTConfig:
        block_size: int = 1024
        batch_size: int = 12
        n_layer: int = 6
        n_head: int = 12
        n_embd: int = 768
        dropout: float = 0.1
        vocab_size: int = 13005
        
        def __post_init__(self):
            self.head_size = self.n_embd // self.n_head

    model = GPT(GPTConfig())
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)

# ---- 2. 加载中文 tokenizer ----
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
tokenizer_path = os.path.join(project_root, "data", "tokenizer_zh_char.pkl")
with open(tokenizer_path, 'rb') as f:
    tk_data = pickle.load(f)

stoi = tk_data['stoi']
itos = tk_data['itos']
unk_id = tk_data['unk_id']
print(f"✅ 已加载中文 tokenizer，词表大小: {tk_data['vocab_size']}")

def encode(text):
    """中文字符级编码"""
    return [stoi.get(ch, unk_id) for ch in text]

def decode(ids):
    """中文字符级解码"""
    return ''.join(itos.get(i, '<unk>') for i in ids)

# ---- 3. 加载 checkpoint ----
ckpt_path = "checkpoints/sft_model_epoch_9.pt"  # 加载最后一个 epoch
if not os.path.exists(ckpt_path):
    # 回退到 epoch_1
    ckpt_path = "checkpoints/model_epoch_0.pt"
ckpt = torch.load(ckpt_path, map_location=device)
# model.load_state_dict(ckpt['model_state_dict'])
model.load_state_dict(ckpt['model'])
model.eval()
print(f"✅ 已加载模型: {ckpt_path}")
print(f"   Epoch: {ckpt['epoch']+1}, 验证 Loss: {ckpt['val_loss']:.4f}")

# ---- 4. 对话函数 ----
def chat(prompt, max_tokens=100, temperature=0.8, top_k=50):
    """用训练好的模型生成中文文本"""
    input_ids = encode(prompt)
    x = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    generated = []
    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            x_cond = x if x.size(1) <= GPTConfig.block_size else x[:, -GPTConfig.block_size:]
            logits, _ = model(x_cond)
            logits = logits[:, -1, :] / temperature

            # Top-K 过滤
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')

            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            x = torch.cat((x, idx_next), dim=1)
            generated.append(idx_next.item())

    return decode(generated)


# ---- 5. 开始聊天 ----
print("\n" + "=" * 60)
print("🤖 模型聊天测试（中文字符级模型）")
print("=" * 60)

test_prompts = ["人工智能是什么","北京","国足","蛇类",]

for prompt in test_prompts:
    print(f"\n💬 输入: {prompt}")
    reply = chat(prompt, max_tokens=128, temperature=0.8, top_k=50)
    print(f"🤖 输出: {reply}")
    print("-" * 40)

print("\n✅ 聊天测试完成！")
print("💡 参数说明: temperature=0.8 (越低越确定), top_k=50 (从概率最高的50个token采样)")
